# GWM-RNN Training on Kaggle - MKG-Y Multimodal Dataset

Comprehensive training experiments for the Graph World Model with RNN (GWM-RNN) on the **MKG-Y multimodal knowledge graph** dataset. MKG-Y is a standard benchmark from YAGO with both textual and visual entity information.

## Dataset
- **Name**: MKG-Y (Multimodal Knowledge Graph - YAGO)
- **Source**: Standard benchmark from published MMKG research
- **Entities**: ~20,000 with text and image embeddings
- **Relations**: Structural embeddings only (no text)
- **Modalities**: Text + Visual + Structural

## Experiments
- **Pooling methods**: Last token, Mean pooling, Max pooling
- **Hyperparameter configurations**: 7 different settings
- **Total experiments**: 21 (3 pooling × 7 configs)

## 1. Install Dependencies

Install required packages if running on Kaggle.

In [ ]:
import os
import sys

# Detect if running on Kaggle
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    print("🔧 Running on Kaggle - Installing dependencies...")
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    !pip install -q transformers sentence-transformers
    !pip install -q scikit-learn numpy pandas matplotlib seaborn
    print("✓ Dependencies installed")
else:
    print("💻 Running locally")

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

Define data paths, hyperparameters, and experiment settings.

In [ ]:
# Data paths
if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/gwm-rnn-kg-mkg-y-multimodal"
    OUTPUT_BASE_DIR = "/kaggle/working/trained_models"
else:
    DATA_DIR = r"d:\NLP research\Code\graph-world-models\GWM\data\mkg-y\processed"
    OUTPUT_BASE_DIR = r"d:\NLP research\Code\graph-world-models\trained\GWM-RNN\link-prediction\MKG-Y"

# Pooling methods to test
POOLING_METHODS = ['last', 'mean', 'max']

# Global settings
SEED = 42

# Experiment control
RUN_ALL_CONFIGS = True  # Set to False to run only one config
SELECTED_CONFIG = "hybrid-negatives"  # Used if RUN_ALL_CONFIGS=False

print("=" * 70)
print("EXPERIMENT CONFIGURATION - MKG-Y MULTIMODAL")
print("=" * 70)
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_BASE_DIR}")
print(f"Pooling methods: {', '.join([p.upper() for p in POOLING_METHODS])}")
print(f"Run all configs: {RUN_ALL_CONFIGS}")
if not RUN_ALL_CONFIGS:
    print(f"Selected config: {SELECTED_CONFIG}")
print(f"Random seed: {SEED}")
print("=" * 70)

In [ ]:
# Hyperparameter sets
HYPERPARAMETER_SETS = [
    {
        'name': 'standard',
        'description': 'Standard multimodal fusion without gating',
        'hidden_dim': 512,
        'fusion_dim': 1024,
        'structural_dim': 768,
        'num_lstm_layers': 2,
        'dropout': 0.2,
        'image_dropout': 0.3,
        'text_dropout': 0.1,
        'batch_size': 256,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_gating': False,
        'use_in_batch_negatives': False,
        'num_epochs': 100,
        'scheduler_patience': 5,
        'early_stopping_patience': 10,
        'eval_every': 1,
        'seed': SEED,
        'num_workers': 2
    },
    {
        'name': 'gating',
        'description': 'Standard multimodal fusion WITH gating mechanism',
        'hidden_dim': 512,
        'fusion_dim': 1024,
        'structural_dim': 768,
        'num_lstm_layers': 2,
        'dropout': 0.2,
        'image_dropout': 0.3,
        'text_dropout': 0.1,
        'batch_size': 256,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_gating': True,
        'use_in_batch_negatives': False,
        'num_epochs': 100,
        'scheduler_patience': 5,
        'early_stopping_patience': 10,
        'eval_every': 1,
        'seed': SEED,
        'num_workers': 2
    },
    {
        'name': 'hybrid-negatives',
        'description': 'RECOMMENDED: Combines sampled + in-batch negatives with gating',
        'hidden_dim': 512,
        'fusion_dim': 1024,
        'structural_dim': 768,
        'num_lstm_layers': 2,
        'dropout': 0.2,
        'image_dropout': 0.3,
        'text_dropout': 0.1,
        'batch_size': 256,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_gating': True,
        'use_in_batch_negatives': True,
        'num_epochs': 100,
        'scheduler_patience': 5,
        'early_stopping_patience': 10,
        'eval_every': 1,
        'seed': SEED,
        'num_workers': 2
    },
    {
        'name': 'in-batch',
        'description': 'Pure in-batch negatives (no sampled negatives)',
        'hidden_dim': 512,
        'fusion_dim': 1024,
        'structural_dim': 768,
        'num_lstm_layers': 2,
        'dropout': 0.2,
        'image_dropout': 0.3,
        'text_dropout': 0.1,
        'batch_size': 256,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'num_negatives': 0,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_gating': True,
        'use_in_batch_negatives': True,
        'num_epochs': 100,
        'scheduler_patience': 5,
        'early_stopping_patience': 10,
        'eval_every': 1,
        'seed': SEED,
        'num_workers': 2
    },
    {
        'name': 'large',
        'description': 'Larger model dimensions with gating',
        'hidden_dim': 768,
        'fusion_dim': 1536,
        'structural_dim': 1024,
        'num_lstm_layers': 2,
        'dropout': 0.3,
        'image_dropout': 0.3,
        'text_dropout': 0.1,
        'batch_size': 128,
        'learning_rate': 0.0005,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_gating': True,
        'use_in_batch_negatives': False,
        'num_epochs': 100,
        'scheduler_patience': 5,
        'early_stopping_patience': 10,
        'eval_every': 1,
        'seed': SEED,
        'num_workers': 2
    },
    {
        'name': 'deep',
        'description': 'Deeper LSTM model (3 layers) with gating',
        'hidden_dim': 512,
        'fusion_dim': 1024,
        'structural_dim': 768,
        'num_lstm_layers': 3,
        'dropout': 0.3,
        'image_dropout': 0.3,
        'text_dropout': 0.1,
        'batch_size': 256,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_gating': True,
        'use_in_batch_negatives': False,
        'num_epochs': 100,
        'scheduler_patience': 5,
        'early_stopping_patience': 10,
        'eval_every': 1,
        'seed': SEED,
        'num_workers': 2
    },
    {
        'name': 'self-adversarial',
        'description': 'RotatE-style self-adversarial negative sampling',
        'hidden_dim': 512,
        'fusion_dim': 1024,
        'structural_dim': 768,
        'num_lstm_layers': 2,
        'dropout': 0.2,
        'image_dropout': 0.3,
        'text_dropout': 0.1,
        'batch_size': 256,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'num_negatives': 128,
        'loss': 'self_adversarial',
        'margin': 9.0,
        'adversarial_temperature': 1.0,
        'distance_based': True,
        'use_gating': True,
        'use_in_batch_negatives': False,
        'num_epochs': 100,
        'scheduler_patience': 5,
        'early_stopping_patience': 10,
        'eval_every': 1,
        'seed': SEED,
        'num_workers': 2
    }
]

# Display configurations
print("=" * 70)
print("HYPERPARAMETER CONFIGURATIONS - MKG-Y")
print("=" * 70)
for i, config in enumerate(HYPERPARAMETER_SETS, 1):
    print(f"\n{i}. {config['name'].upper()}")
    print(f"   {config['description']}")
    print(f"   • Hidden: {config['hidden_dim']}, Fusion: {config['fusion_dim']}, Structural: {config['structural_dim']}")
    print(f"   • LSTM layers: {config['num_lstm_layers']}, Dropout: {config['dropout']}")
    print(f"   • Gating: {'Yes' if config.get('use_gating', False) else 'No'}")
    print(f"   • Loss: {config['loss']}, Negatives: {config['num_negatives']}")
    if config.get('use_in_batch_negatives', False):
        print(f"   • In-batch negatives: Yes")

print(f"\n{'=' * 70}")
print(f"Total experiments: {len(POOLING_METHODS)} pooling × {len(HYPERPARAMETER_SETS)} configs = {len(POOLING_METHODS) * len(HYPERPARAMETER_SETS)}")
print(f"{'=' * 70}")

## 3. Copy Training Files from GitHub

If running on Kaggle, copy the training scripts from GitHub.

In [ ]:
required_files = ['model.py', 'dataset.py', 'train.py', 'utils.py', 'generate_negatives.py']

if IS_KAGGLE:
    print("="*70)
    print("Cloning GitHub repository...")
    print("="*70)
    
    # Clone your GitHub repo
    GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
    BRANCH = " no-rel-text"
    
    !git clone {GITHUB_REPO} /kaggle/working/gwm
    %cd /kaggle/working/gwm
    !git checkout {BRANCH}
    !git pull
    %cd ../
    
    # Copy files from repo to working directory
    repo_path = "/kaggle/working/gwm/gwm-rnn/link-prediction/multimodal"
    
    print(f"\nCopying files from {repo_path}...")
    for file in required_files:
        !cp {repo_path}/{file} /kaggle/working/
        print(f"✓ Copied {file}")
else:
    print("Running locally - files should be in parent directory")
    # Add parent directory to path for local imports
    import sys
    from pathlib import Path
    parent_dir = str(Path(__file__).parent.parent) if '__file__' in globals() else r'D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\link-prediction\multimodal'
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)
    print(f"Added to path: {parent_dir}")

# Verify files exist
import os
if IS_KAGGLE:
    base_path = '/kaggle/working'
else:
    base_path = r'D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\link-prediction\multimodal'

missing_files = []
for file in required_files:
    file_path = os.path.join(base_path, file) if IS_KAGGLE else os.path.join(base_path, file)
    if not os.path.exists(file_path):
        missing_files.append(file)

if missing_files:
    print(f"\n⚠️  Some files not found: {missing_files}")
    print("Training scripts will be imported from parent directory")
else:
    print(f"\n✓ All required files ready: {required_files}")

## 4. Generate Fixed Negative Samples

Pre-generate negative samples to ensure consistent evaluation across experiments.

In [ ]:
import torch
from pathlib import Path

# On Kaggle, save to working directory (input is read-only)
if IS_KAGGLE:
    negatives_output_dir = Path('/kaggle/working')
else:
    negatives_output_dir = Path(DATA_DIR)

negatives_path = negatives_output_dir / 'train_negatives.pt'

# Also check if negatives exist in DATA_DIR (for pre-uploaded datasets)
data_dir_negatives = Path(DATA_DIR) / 'train_negatives.pt'

# Determine max negatives needed across all configs
configs_needing_negatives = [config['num_negatives'] for config in HYPERPARAMETER_SETS if config['num_negatives'] > 0]

if not configs_needing_negatives:
    print("=" * 70)
    print("ℹ️  All configurations use in-batch negatives")
    print("   No fixed negative generation needed")
    print("=" * 70)
else:
    max_negatives = max(configs_needing_negatives)
    print(f"ℹ️  Maximum negatives needed: {max_negatives}")
    
    if negatives_path.exists():
        print("=" * 70)
        print("✓ Fixed negatives already exist")
        print("=" * 70)
        print(f"Location: {negatives_path}")
        
        # Load and verify
        train_negatives = torch.load(negatives_path, map_location='cpu')
        print(f"Shape: {train_negatives.shape}")
        print(f"Size: {negatives_path.stat().st_size / (1024**2):.2f} MB")
        
        # Check if we have enough negatives
        if train_negatives.shape[1] < max_negatives:
            print()
            print(f"⚠️  WARNING: Existing negatives have {train_negatives.shape[1]} samples")
            print(f"   but {max_negatives} are needed for some configs.")
            print(f"   Regenerating with {max_negatives} negatives...")
            print("=" * 70)
            
            # Regenerate with more negatives
            !python generate_negatives.py \
                --data_dir {DATA_DIR} \
                --output_dir {negatives_output_dir} \
                --num_negatives {max_negatives} \
                --seed {SEED} \
                --force
        else:
            print()
            print("All experiments will use these identical negative samples.")
            print(f"(Each experiment will use the first N negatives as needed)")
            print("=" * 70)
    elif data_dir_negatives.exists():
        print("=" * 70)
        print("✓ Fixed negatives found in dataset")
        print("=" * 70)
        print(f"Location: {data_dir_negatives}")
        
        # Load and check
        train_negatives = torch.load(data_dir_negatives, map_location='cpu')
        print(f"Shape: {train_negatives.shape}")
        
        if train_negatives.shape[1] < max_negatives:
            print()
            print(f"⚠️  WARNING: Existing negatives have {train_negatives.shape[1]} samples")
            print(f"   but {max_negatives} are needed. Regenerating...")
            print("=" * 70)
            
            !python generate_negatives.py \
                --data_dir {DATA_DIR} \
                --output_dir {negatives_output_dir} \
                --num_negatives {max_negatives} \
                --seed {SEED} \
                --force
        else:
            # Copy to working directory for consistency
            import shutil
            shutil.copy(data_dir_negatives, negatives_path)
            print(f"Copied to: {negatives_path}")
            print(f"Size: {negatives_path.stat().st_size / (1024**2):.2f} MB")
            print()
            print("All experiments will use these identical negative samples.")
            print("=" * 70)
    else:
        print("=" * 70)
        print("GENERATING FIXED NEGATIVE SAMPLES")
        print("=" * 70)
        print()
        print(f"Generating {max_negatives} negatives per triple for all experiments")
        print(f"(Each config will use the first N as needed)")
        print()
        
        # Run the generation script with output to working directory
        !python generate_negatives.py \
            --data_dir {DATA_DIR} \
            --output_dir {negatives_output_dir} \
            --num_negatives {max_negatives} \
            --seed {SEED}
        
        print()
        print("=" * 70)
        print("GENERATION COMPLETE")
        print("=" * 70)

## 5. Run Training Experiments

Train models with all pooling methods and multimodal configurations on MKG-Y.

In [ ]:
import time
from datetime import datetime
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Determine which configs to run
if RUN_ALL_CONFIGS:
    configs_to_run = HYPERPARAMETER_SETS
else:
    configs_to_run = [c for c in HYPERPARAMETER_SETS if c['name'] == SELECTED_CONFIG]

# Track all experiment results
all_results = []
experiment_start_time = time.time()

print("="*80)
print(" "*20 + "STARTING MKG-Y MULTIMODAL EXPERIMENTS")
print("="*80)
print(f"\nTotal experiments to run: {len(POOLING_METHODS)} × {len(configs_to_run)} = {len(POOLING_METHODS) * len(configs_to_run)}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

experiment_num = 0
total_experiments = len(POOLING_METHODS) * len(configs_to_run)

for config in configs_to_run:
    for pooling in POOLING_METHODS:
        experiment_num += 1
        
        print(f"\n{'='*80}")
        print(f" EXPERIMENT {experiment_num}/{total_experiments}: {config['name'].upper()} + {pooling.upper()}-POOLING (MKG-Y)")
        print(f"{'='*80}")
        
        # Create output directory
        output_dir = f"{OUTPUT_BASE_DIR}/{config['name']}/{pooling}-pooling"
        
        # Build training command with ALL parameters
        if config['loss'] == 'infonce':
            loss_args = f"--loss infonce --temperature {config.get('temperature', 0.07)}"
            if config.get('use_in_batch_negatives', False):
                loss_args += " --use_in_batch_negatives"
        elif config['loss'] in ['self_adversarial', 'self_adversarial_margin']:
            loss_args = f"--loss {config['loss']} --margin {config.get('margin', 9.0)}"
            loss_args += f" --adversarial_temperature {config.get('adversarial_temperature', 1.0)}"
            if config.get('distance_based', False):
                loss_args += " --distance_based"
        else:
            loss_args = f"--loss margin --margin {config.get('margin', 1.0)}"
        
        # Gating mechanism
        gating_args = "--use_gating" if config.get('use_gating', False) else ""
        
        cmd = f"""python train.py \
            --data_dir {DATA_DIR} \
            --output_dir {output_dir} \
            --hidden_dim {config['hidden_dim']} \
            --fusion_dim {config.get('fusion_dim', 1024)} \
            --structural_dim {config.get('structural_dim', 768)} \
            --num_lstm_layers {config['num_lstm_layers']} \
            --dropout {config['dropout']} \
            --image_dropout {config.get('image_dropout', 0.3)} \
            --text_dropout {config.get('text_dropout', 0.1)} \
            --pooling {pooling} \
            --num_epochs {config.get('num_epochs', 100)} \
            --batch_size {config['batch_size']} \
            --learning_rate {config['learning_rate']} \
            --weight_decay {config.get('weight_decay', 1e-4)} \
            --max_grad_norm {config.get('max_grad_norm', 1.0)} \
            --num_negatives {config['num_negatives']} \
            {loss_args} \
            {gating_args} \
            --scheduler_patience {config.get('scheduler_patience', 5)} \
            --early_stopping_patience {config.get('early_stopping_patience', 10)} \
            --eval_every {config.get('eval_every', 1)} \
            --seed {config.get('seed', 42)} \
            --num_workers {config.get('num_workers', 2)}"""
        
        print(f"\n📋 Configuration:")
        print(f"   Dataset: MKG-Y Multimodal (~20,000 entities, YAGO relations)")
        print(f"   Config: {config['name']} - {config['description']}")
        print(f"   Pooling: {pooling}")
        print(f"\n   Model Architecture:")
        print(f"   • Hidden dim: {config['hidden_dim']}")
        print(f"   • Fusion dim: {config.get('fusion_dim', 1024)}")
        print(f"   • Structural dim: {config.get('structural_dim', 768)}")
        print(f"   • LSTM layers: {config['num_lstm_layers']}")
        print(f"   • Dropout: {config['dropout']}")
        print(f"   • Image dropout: {config.get('image_dropout', 0.3)}")
        print(f"   • Text dropout: {config.get('text_dropout', 0.1)}")
        print(f"   • Gating: {'Yes' if config.get('use_gating', False) else 'No'}")
        print(f"\n   Training:")
        print(f"   • Epochs: {config.get('num_epochs', 100)}")
        print(f"   • Batch size: {config['batch_size']}")
        print(f"   • Learning rate: {config['learning_rate']}")
        print(f"   • Weight decay: {config.get('weight_decay', 1e-4)}")
        print(f"   • Gradient clipping: {config.get('max_grad_norm', 1.0)}")
        print(f"\n   Loss & Negatives:")
        print(f"   • Loss: {config['loss']}")
        if config.get('use_in_batch_negatives', False):
            print(f"   • Negatives: In-batch ({config['batch_size']-1} per sample)")
        else:
            print(f"   • Negatives: {config['num_negatives']} sampled")
        if config['loss'] == 'infonce':
            print(f"   • Temperature: {config.get('temperature', 0.07)}")
        elif 'adversarial' in config['loss']:
            print(f"   • Margin: {config.get('margin', 9.0)}")
            print(f"   • Adversarial temp: {config.get('adversarial_temperature', 1.0)}")
            print(f"   • Distance-based: {config.get('distance_based', False)}")
        print(f"\n   Optimization:")
        print(f"   • Scheduler patience: {config.get('scheduler_patience', 5)}")
        print(f"   • Early stopping patience: {config.get('early_stopping_patience', 10)}")
        print(f"   • Eval every: {config.get('eval_every', 1)} epoch(s)")
        print(f"\n   System:")
        print(f"   • Seed: {config.get('seed', 42)}")
        print(f"   • Workers: {config.get('num_workers', 2)}")
        print(f"   • Output: {output_dir}")
        
        print(f"\n🚀 Starting training...")
        print("-"*80)
        
        # Execute training
        !{cmd}
        
        # Load and store results
        try:
            result_path = Path(output_dir) / "test_results.json"
            history_path = Path(output_dir) / "training_history.json"
            
            if result_path.exists() and history_path.exists():
                with open(result_path) as f:
                    test_results = json.load(f)
                with open(history_path) as f:
                    history = json.load(f)
                
                test_metrics = test_results['test_metrics']
                train_time = sum(history['epoch_times'])
                
                all_results.append({
                    'config_name': config['name'],
                    'pooling': pooling,
                    'hidden_dim': config['hidden_dim'],
                    'fusion_dim': config.get('fusion_dim', 1024),
                    'structural_dim': config.get('structural_dim', 768),
                    'num_layers': config['num_lstm_layers'],
                    'dropout': config['dropout'],
                    'image_dropout': config.get('image_dropout', 0.3),
                    'text_dropout': config.get('text_dropout', 0.1),
                    'loss': config['loss'],
                    'use_gating': config.get('use_gating', False),
                    'learning_rate': config['learning_rate'],
                    'weight_decay': config.get('weight_decay', 1e-4),
                    'batch_size': config['batch_size'],
                    'num_negatives': config['num_negatives'],
                    'num_epochs': config.get('num_epochs', 100),
                    'test_mrr': test_metrics['MRR'],
                    'test_mr': test_metrics['MR'],
                    'test_hits@1': test_metrics['Hits@1'],
                    'test_hits@3': test_metrics['Hits@3'],
                    'test_hits@10': test_metrics['Hits@10'],
                    'test_hits@50': test_metrics['Hits@50'],
                    'best_val_mrr': test_results['best_val_mrr'],
                    'best_epoch': test_results['best_epoch'],
                    'training_time': train_time,
                    'output_dir': output_dir
                })
                
                print(f"\n✅ Experiment {experiment_num} completed!")
                print(f"   Test MRR: {test_metrics['MRR']:.4f}")
                print(f"   Test Hits@10: {test_metrics['Hits@10']:.4f} ({test_metrics['Hits@10']*100:.2f}%)")
                print(f"   Training time: {train_time:.1f}s ({train_time/60:.1f} min)")
            else:
                print(f"\n⚠️  Results not found for experiment {experiment_num}")
        except Exception as e:
            print(f"\n❌ Error loading results: {e}")
        
        print(f"\n{'='*80}\n")

total_time = time.time() - experiment_start_time

print(f"\n{'='*80}")
print(" "*20 + "ALL MKG-Y EXPERIMENTS COMPLETED")
print(f"{'='*80}")
print(f"Total experiments: {len(all_results)}/{total_experiments}")
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
if len(all_results) > 0:
    print(f"Average time per experiment: {total_time/len(all_results)/60:.1f} minutes")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

## 6. Comprehensive Results Analysis

Analyze and compare all MKG-Y multimodal experiments.

In [ ]:
if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    print("="*80)
    print(" "*15 + "MKG-Y MULTIMODAL EXPERIMENT RESULTS")
    print("="*80)
    
    # Overall best
    best_idx = df_results['test_mrr'].idxmax()
    best_result = df_results.loc[best_idx]
    
    print(f"\n🏆 BEST OVERALL PERFORMANCE:")
    print(f"   Config: {best_result['config_name']} + {best_result['pooling']}-pooling")
    print(f"   Gating: {'Yes' if best_result['use_gating'] else 'No'}")
    print(f"   Test MRR: {best_result['test_mrr']:.4f}")
    print(f"   Test Hits@1: {best_result['test_hits@1']:.4f} ({best_result['test_hits@1']*100:.2f}%)")
    print(f"   Test Hits@10: {best_result['test_hits@10']:.4f} ({best_result['test_hits@10']*100:.2f}%)")
    print(f"   Training time: {best_result['training_time']/60:.1f} min")
    
    # Pooling method comparison
    print(f"\n📊 PERFORMANCE BY POOLING METHOD:")
    print("-"*80)
    pooling_summary = df_results.groupby('pooling').agg({
        'test_mrr': ['mean', 'std', 'max'],
        'test_hits@10': ['mean', 'max'],
        'training_time': 'mean'
    }).round(4)
    
    for pooling in POOLING_METHODS:
        if pooling in pooling_summary.index:
            row = pooling_summary.loc[pooling]
            print(f"\n{pooling.upper()}-Pooling:")
            print(f"   Avg MRR: {row[('test_mrr', 'mean')]:.4f} ± {row[('test_mrr', 'std')]:.4f}")
            print(f"   Max MRR: {row[('test_mrr', 'max')]:.4f}")
            print(f"   Avg Hits@10: {row[('test_hits@10', 'mean')]:.4f}")
            print(f"   Avg Time: {row[('training_time', 'mean')]/60:.1f} min")
    
    # Modality comparison
    print(f"\n🔍 PERFORMANCE BY FUSION METHOD:")
    print("-"*80)
    
    with_gating = df_results[df_results['use_gating'] == True]
    without_gating = df_results[df_results['use_gating'] == False]
    
    if len(without_gating) > 0:
        print(f"\nWithout Gating:")
        print(f"   Avg MRR: {without_gating['test_mrr'].mean():.4f}")
        print(f"   Max MRR: {without_gating['test_mrr'].max():.4f}")
        print(f"   Avg Hits@10: {without_gating['test_hits@10'].mean():.4f}")
    
    if len(with_gating) > 0:
        print(f"\nWith Gating Mechanism:")
        print(f"   Avg MRR: {with_gating['test_mrr'].mean():.4f}")
        print(f"   Max MRR: {with_gating['test_mrr'].max():.4f}")
        print(f"   Avg Hits@10: {with_gating['test_hits@10'].mean():.4f}")
        
        # Calculate improvement
        if len(without_gating) > 0:
            improvement = ((with_gating['test_mrr'].mean() - without_gating['test_mrr'].mean()) / without_gating['test_mrr'].mean()) * 100
            print(f"   Improvement: {improvement:+.2f}%")
    
    # Detailed results table
    print(f"\n📋 DETAILED RESULTS TABLE:")
    print("-"*80)
    display_cols = ['config_name', 'pooling', 'fusion_dim', 'use_gating', 
                    'test_mrr', 'test_hits@10', 'training_time']
    df_display = df_results[display_cols].copy()
    df_display['test_mrr'] = df_display['test_mrr'].apply(lambda x: f"{x:.4f}")
    df_display['test_hits@10'] = df_display['test_hits@10'].apply(lambda x: f"{x:.4f}")
    df_display['training_time'] = df_display['training_time'].apply(lambda x: f"{x/60:.1f}min")
    
    print(df_display.to_string(index=False))
    
    # Save results
    results_csv_path = f"{OUTPUT_BASE_DIR}/mkg-y_multimodal_results.csv"
    df_results.to_csv(results_csv_path, index=False)
    print(f"\n✓ Saved results to: {results_csv_path}")
    
    print("\n" + "="*80)
else:
    print("❌ No results available.")

## 7. Visualization

Create visualizations comparing different configurations and fusion strategies.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    sns.set_style("whitegrid")
    
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('GWM-RNN Multimodal Performance on MKG-Y', fontsize=16, fontweight='bold')
    
    # 1. MRR by pooling method
    ax1 = plt.subplot(2, 3, 1)
    pooling_data = df_results.groupby('pooling')['test_mrr'].apply(list)
    bp1 = ax1.boxplot([pooling_data[p] for p in POOLING_METHODS if p in pooling_data.index],
                       labels=[p.upper() for p in POOLING_METHODS if p in pooling_data.index],
                       patch_artist=True)
    for patch in bp1['boxes']:
        patch.set_facecolor('lightblue')
    ax1.set_ylabel('Test MRR', fontsize=11, fontweight='bold')
    ax1.set_title('MRR by Pooling Method', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 2. Gating mechanism comparison
    ax2 = plt.subplot(2, 3, 2)
    gating_data = []
    gating_labels = []
    
    without_gating = df_results[df_results['use_gating'] == False]
    with_gating = df_results[df_results['use_gating'] == True]
    
    if len(without_gating) > 0:
        gating_data.append(without_gating['test_mrr'].values)
        gating_labels.append('No Gating')
    if len(with_gating) > 0:
        gating_data.append(with_gating['test_mrr'].values)
        gating_labels.append('With Gating')
    
    if gating_data:
        bp2 = ax2.boxplot(gating_data, labels=gating_labels, patch_artist=True)
        colors = ['lightcoral', 'lightgreen']
        for patch, color in zip(bp2['boxes'], colors[:len(gating_data)]):
            patch.set_facecolor(color)
    ax2.set_ylabel('Test MRR', fontsize=11, fontweight='bold')
    ax2.set_title('MRR by Fusion Strategy', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. Hits@K comparison
    ax3 = plt.subplot(2, 3, 3)
    hits_cols = ['test_hits@1', 'test_hits@3', 'test_hits@10']
    hits_means = df_results[hits_cols].mean()
    x = np.arange(len(hits_means))
    bars = ax3.bar(x, hits_means.values, color=['#ff9999', '#66b3ff', '#99ff99'])
    ax3.set_xticks(x)
    ax3.set_xticklabels(['Hits@1', 'Hits@3', 'Hits@10'])
    ax3.set_ylabel('Average Score', fontsize=11, fontweight='bold')
    ax3.set_title('Average Hits@K Scores', fontsize=12, fontweight='bold')
    for bar, val in zip(bars, hits_means.values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. MRR vs Training Time
    ax4 = plt.subplot(2, 3, 4)
    colors = {'last': 'blue', 'mean': 'green', 'max': 'red'}
    for pooling in POOLING_METHODS:
        mask = df_results['pooling'] == pooling
        ax4.scatter(df_results[mask]['training_time']/60, 
                   df_results[mask]['test_mrr'],
                   c=colors.get(pooling, 'gray'),
                   label=pooling.upper(),
                   s=100, alpha=0.6, edgecolors='black')
    ax4.set_xlabel('Training Time (minutes)', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Test MRR', fontsize=11, fontweight='bold')
    ax4.set_title('MRR vs Training Time', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Configuration comparison
    ax5 = plt.subplot(2, 3, 5)
    config_mrr = df_results.groupby('config_name')['test_mrr'].mean().sort_values(ascending=False)
    bars = ax5.barh(range(len(config_mrr)), config_mrr.values)
    ax5.set_yticks(range(len(config_mrr)))
    ax5.set_yticklabels([name[:20] for name in config_mrr.index], fontsize=9)
    ax5.set_xlabel('Average Test MRR', fontsize=11, fontweight='bold')
    ax5.set_title('MRR by Configuration', fontsize=12, fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='x')
    
    # 6. Loss function comparison
    ax6 = plt.subplot(2, 3, 6)
    if 'loss' in df_results.columns:
        loss_mrr = df_results.groupby('loss')['test_mrr'].mean().sort_values(ascending=False)
        if len(loss_mrr) > 1:
            bars = ax6.bar(range(len(loss_mrr)), loss_mrr.values, 
                          color=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'][:len(loss_mrr)])
            ax6.set_xticks(range(len(loss_mrr)))
            ax6.set_xticklabels(loss_mrr.index, rotation=45, ha='right')
            ax6.set_ylabel('Average Test MRR', fontsize=11, fontweight='bold')
            ax6.set_title('MRR by Loss Function', fontsize=12, fontweight='bold')
            for bar, val in zip(bars, loss_mrr.values):
                ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax6.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    viz_path = f"{OUTPUT_BASE_DIR}/mkg-y_multimodal_comparison.png"
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved visualization to: {viz_path}")
else:
    print("❌ No results to visualize.")

## 8. Best Model Analysis

Detailed analysis of the best performing model configuration.

In [ ]:
if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    print("=" * 80)
    print("BEST MULTIMODAL MODEL ANALYSIS - MKG-Y")
    print("=" * 80)
    print()
    
    best_idx = df_results['test_mrr'].idxmax()
    best_result = df_results.iloc[best_idx]
    
    print("🏆 BEST MODEL CONFIGURATION:")
    print(f"  • Configuration: {best_result['config_name'].upper()}")
    print(f"  • Pooling Method: {best_result['pooling'].upper()}")
    print(f"  • Gating Mechanism: {'Yes' if best_result['use_gating'] else 'No'}")
    print(f"  • Hidden Dimension: {best_result['hidden_dim']}")
    print(f"  • Fusion Dimension: {best_result['fusion_dim']}")
    print(f"  • LSTM Layers: {best_result['num_layers']}")
    print(f"  • Loss Function: {best_result['loss'].upper()}")
    print()
    
    print("📊 PERFORMANCE METRICS:")
    print(f"  • Test MRR:       {best_result['test_mrr']:.4f}")
    print(f"  • Test MR:        {best_result['test_mr']:.2f}")
    print(f"  • Test Hits@1:    {best_result['test_hits@1']:.4f} ({best_result['test_hits@1']*100:.2f}%)")
    print(f"  • Test Hits@3:    {best_result['test_hits@3']:.4f} ({best_result['test_hits@3']*100:.2f}%)")
    print(f"  • Test Hits@10:   {best_result['test_hits@10']:.4f} ({best_result['test_hits@10']*100:.2f}%)")
    print()
    
    print("⏱️  TRAINING EFFICIENCY:")
    print(f"  • Training Time:  {best_result['training_time']/60:.1f} min")
    print(f"  • Best Epoch:     {best_result['best_epoch']}")
    print()
    
    # Gating benefit analysis
    no_gating_results = df_results[df_results['use_gating'] == False]
    if len(no_gating_results) > 0 and best_result['use_gating']:
        avg_no_gating_mrr = no_gating_results['test_mrr'].mean()
        improvement = ((best_result['test_mrr'] - avg_no_gating_mrr) / avg_no_gating_mrr) * 100
        print("🎯 GATING MECHANISM BENEFIT:")
        print(f"  • No gating baseline MRR: {avg_no_gating_mrr:.4f}")
        print(f"  • Best with gating MRR:   {best_result['test_mrr']:.4f}")
        print(f"  • Improvement:            {improvement:+.2f}%")
        print()
    
    print("📈 POOLING METHOD RANKING:")
    pooling_ranking = df_results.groupby('pooling')['test_mrr'].mean().sort_values(ascending=False)
    for rank, (pooling, mrr) in enumerate(pooling_ranking.items(), 1):
        print(f"  {rank}. {pooling.upper()}-pooling: {mrr:.4f}")
    
    print("\n" + "=" * 80)
else:
    print("❌ No results available.")

## 9. Download Results

Summary of all output files.

In [ ]:
print("=" * 80)
print("EXPERIMENT OUTPUT SUMMARY - MKG-Y MULTIMODAL")
print("=" * 80)
print()
print(f"📁 Output directory: {OUTPUT_BASE_DIR}")
print()

if os.path.exists(OUTPUT_BASE_DIR):
    print("📂 EXPERIMENT DIRECTORIES:")
    for config_name in os.listdir(OUTPUT_BASE_DIR):
        config_path = os.path.join(OUTPUT_BASE_DIR, config_name)
        if os.path.isdir(config_path) and not config_name.startswith('.'):
            print(f"\n  • {config_name}/")
            for pooling_dir in os.listdir(config_path):
                pooling_path = os.path.join(config_path, pooling_dir)
                if os.path.isdir(pooling_path):
                    print(f"    └── {pooling_dir}/")
                    files = os.listdir(pooling_path)
                    for file in sorted(files):
                        if file.endswith(('.pt', '.json', '.png')):
                            file_path = os.path.join(pooling_path, file)
                            size_mb = os.path.getsize(file_path) / (1024 * 1024)
                            print(f"        ├── {file} ({size_mb:.2f} MB)")
    
    print()
    print("📊 SUMMARY FILES:")
    summary_files = ['mkg-y_multimodal_results.csv', 'mkg-y_multimodal_comparison.png']
    for file in summary_files:
        file_path = os.path.join(OUTPUT_BASE_DIR, file)
        if os.path.exists(file_path):
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  ✓ {file} ({size_mb:.2f} MB)")
    
    print()
    
    if len(all_results) > 0:
        df_results = pd.DataFrame(all_results)
        best_idx = df_results['test_mrr'].idxmax()
        best_result = df_results.iloc[best_idx]
        
        print("🏆 BEST MODEL FILES:")
        print(f"  {best_result['output_dir']}/")
        print(f"  • best_model.pt")
        print(f"  • training_history.json")
        print(f"  • test_results.json")
        print()
        
        print("FINAL SUMMARY:")
        print(f"  • Total experiments: {len(all_results)}")
        print(f"  • Best MRR: {df_results['test_mrr'].max():.4f}")
        print(f"  • Best Hits@10: {df_results['test_hits@10'].max():.4f}")
    
    print("=" * 80)
else:
    print("Output directory not found. Run experiments first.")